# acttrace — a tamper-evident record of what the agent did

"Prove what the agent saw and decided" is a real ask from compliance, from security, and from your own future self. `acttrace` hash-chains every event; `verify()` re-walks the chain, and one edited byte breaks it at a known sequence number.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · Record a decision

A decision scope carries the model call **and** the human sign-off, so the record answers *who approved this* as well as *what ran*.

In [ ]:
import os
import pathlib
import tempfile

from cendor.acttrace import AuditLog, verify

SIGNING_KEY = os.environ.get("CENDOR_DEMO_KEY", "demo-signing-key")
d = tempfile.mkdtemp()
raw, evidence = str(pathlib.Path(d) / "audit.jsonl"), str(pathlib.Path(d) / "evidence.jsonl")

audit = AuditLog(system="support_bot", risk_tier="limited", path=raw, signing_key=SIGNING_KEY)
with audit.decision(input="summarize the quarterly refunds report", actor="agent") as d1:
    d1.record(model="gpt-4o", prompt_id="summarize@v2")
    d1.human_oversight(reviewer="ops@acme", action="approved", note="spot-checked output")
audit.export(evidence, framework="eu_ai_act")
audit.detach()  # flush + close before reading it back
len(audit.entries)

## 2 · Verify the clean pack

In [ ]:
ok, detail = verify(evidence, key=SIGNING_KEY)
print(f"verify: {ok}  ({detail})")

## 3 · Flip one byte

Not a deleted line, not a reordered file — one character inside a payload that is hashed into the chain. A chain that only caught coarse edits would not be worth much.

In [ ]:
data = pathlib.Path(evidence).read_bytes()
i = data.index(b"quarterly")
pathlib.Path(evidence).write_bytes(data[:i] + b"Q" + data[i + 1 :])
ok2, detail2 = verify(evidence, key=SIGNING_KEY)
print(f"verify: {ok2}  ({detail2})")

## 4 · Prove it

⚠️ The head hash is **per-run** — entries carry timestamps, so it changes every time. The entry count and `verify()` reproduce; the hash does not.

In [ ]:
assert ok and not ok2, "clean log verifies; tampered log must fail"
print("OK")